# Interacting Hamiltonian: Result Analysis

## Tomography Analysis: Density Matrix
The density matrix provides a complete mathematical description of the
quantum state of the system. Its diagonal elements represent the
probabilities of the different quantum states, while the off-diagonal
elements describe quantum coherence and correlations between them.

By reconstructing the density matrix at each evolution step, we can observe
how the populations, coherence, and correlations of the quantum system change
over time.

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.quantum_info import DensityMatrix, Pauli
from qiskit.visualization import plot_state_city

service = QiskitRuntimeService(channel="ibm_quantum_platform")
job = service.job("")  # replace with printed JOB ID
result = job.result()

print("Job loaded")
print("Total circuits:", len(result))

# FULL TOMOGRAPHY BASES (15)
bases = [
    "ZZ","ZX","ZY","ZI",
    "XZ","XX","XY","XI",
    "YZ","YX","YY","YI",
    "IZ","IX","IY"
]

# SAFE COUNTS EXTRACTOR
def get_counts(pub):
    try:
        return pub.data.meas.get_counts()
    except:
        bitstrings = pub.data.meas.get_bitstrings()
        counts = {}
        for b in bitstrings:
            counts[b] = counts.get(b, 0) + 1
        return counts

# EXPECTATION VALUE
def expectation_from_counts(counts):
    shots = sum(counts.values())
    val = 0.0
    for bit, c in counts.items():
        parity = (-1)**(bit.count("1"))
        val += parity * c / shots
    return val

# RECONSTRUCT RHO
def reconstruct_density_matrix(result, iteration):
    start = iteration * len(bases)
    expvals = {}

    for i, basis in enumerate(bases):
        counts = get_counts(result._pub_results[start + i])
        expvals[basis] = expectation_from_counts(counts)

    rho = np.zeros((4,4), dtype=complex)

    for a in "IXYZ":
        for b in "IXYZ":
            if a == "I" and b == "I":
                coeff = 1.0
            else:
                coeff = expvals.get(a + b, 0.0)
            rho += coeff * Pauli(a + b).to_matrix()

    return rho / 4

# DETECT ITERATIONS
num_steps = len(result) // len(bases)
print("Detected iterations:", num_steps)

for step in range(num_steps):
    rho = reconstruct_density_matrix(result, step)

    print(f"\nIteration {step+1} density matrix:\n")
    print(np.round(rho, 3))

    fig = plot_state_city(DensityMatrix(rho))
    display(fig)

    if step == 0: fig.savefig("iteration_1_density_matrix.png", dpi=300)
    if step == 4: fig.savefig("iteration_5_density_matrix.png", dpi=300)
    
    plt.close(fig)

## Fidelity Analysis: State Accuracy

Quantum state fidelity measures how closely the experimentally reconstructed
quantum state matches the corresponding ideal state. A fidelity value close to
$1$ indicates strong agreement, while lower values indicate deviations caused
by hardware noise, gate errors, and measurement imperfections.

By calculating the fidelity at each evolution step, we can evaluate how
accurately the experimental quantum state follows the ideal time evolution.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import sqrtm

fidelity_dmi = []
iterations_axis = []

for i in range(iterations):

    rho_real = hardware_density_matrix[i]
    rho_ideal = ideal_density_matrix[i].data

    # Optional stabilization (recommended for hardware tomography)
    rho_real = (rho_real + rho_real.conj().T) / 2
    rho_real = rho_real / np.trace(rho_real)

    # Uhlmann Fidelity
    sqrt_rho = sqrtm(rho_real)
    inner = sqrt_rho @ rho_ideal @ sqrt_rho
    F = np.trace(sqrtm(inner))**2
    F = np.real(F)
    F = np.clip(F, 0, 1)

    fidelity_dmi.append(F)
    iterations_axis.append(i + 1)

print("Iterations:", iterations_axis)
print("Fidelities:", fidelity_dmi)
fig = plt.figure(figsize=(9.5,6))
plt.plot(iterations_axis, fidelity_dmi, marker='o',color = 'b')
plt.xlabel("Iteration", fontsize=18)
plt.ylabel("Fidelity", fontsize=18)
plt.title("DMI Model", fontsize = 18)
plt.xticks(iterations_axis, fontsize=18)
plt.yticks(fontsize=18)
fig.savefig("dhdj.png", dpi=300)
plt.show()

### Caution
Before this code is executed, the ideal and real density matrices should be constructed and assigned their appropriate names. These matrices should then be inserted here so that they can be compared and analyzed.

## Time-Evolution Analysis: State Progress

The time evolution of the quantum system is analysed by comparing the ideal and
experimentally reconstructed density matrices at each evolution step. This
allows us to examine how the quantum state changes over time and how closely
the experimental evolution follows the expected ideal dynamics.

For each time step $t$, the ideal density matrix
$\rho_{\mathrm{ideal}}(t)$ is compared with the corresponding real density
matrix $\rho_{\mathrm{real}}(t)$. By analysing these density matrices across
all evolution steps, we can identify deviations in the experimental dynamics
and assess the effect of noise, gate errors, and other imperfections on the
time evolution of the system.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit_ibm_runtime import QiskitRuntimeService

# Parameters
dt = # Time Interval
time_steps = # Time Duration
shots = # Shots
states = ["00", "01", "10", "11"]
run_results = np.zeros((4, time_steps))

# IBM Runtime service
service = QiskitRuntimeService(channel="ibm_quantum_platform")

# Your job ID
job = service.job("")
result = job.result()

# Extract probabilities
for t_idx in range(time_steps):
    pub_result = result[t_idx]
    counts = pub_result.data.meas.get_counts()

    for s_idx, state in enumerate(states):
        run_results[s_idx, t_idx] = counts.get(state, 0) / shots

# Time axis
times = np.arange(1, time_steps + 1) * dt

fig = plt.figure(figsize=(10, 6))

for i, state in enumerate(states):
    plt.plot(times, run_results[i], marker='o', linewidth=2, label=f"|{state}⟩")

plt.xlabel("Time")
plt.ylabel("Probability")
plt.title("DMI Time Evolution (IBM FEZ)")
plt.legend()
plt.grid(True)
plt.tight_layout()
fig.savefig("DMI Fidelity.png", dpi=300)
plt.show()